# Stencil Computation for 2D Gray-Scott Using Julia — Proposed Solution



# 1. Installation and Setup



JuliaGPU packages are easy to install: Just do `Pkg.add("Package Name")` to install the package.



In [ ]:
using Pkg
Pkg.add(["BenchmarkTools", "ProgressMeter", "Plots", "ColorTypes", "KernelAbstractions", "CUDA", "AMDGPU", "Metal"])



In [ ]:
using BenchmarkTools, ProgressMeter, Plots, ColorTypes, KernelAbstractions

In [ ]:
Pkg.status()



## 1.1. Selecting a backend

GPU backend packages provide bindings to NVIDIA's CUDA or respective other vendor. CUDA.jl provides all of the compiler and runtime logic needed to program NVIDIA GPUs; the only thing you need to provide is a functional NVIDIA driver (which most HPC systems already have installed and configured), but you don't need to install the CUDA toolkit! CUDA.jl downloads one if it's not already available on your system (again, HPC systems usually have this provided for you):



In [ ]:
# add all options for different backends and array types
DEV = :NVIDIA

if DEV == :NVIDIA
    using CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMD
    using AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    using Metal
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
else DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
end



# 2. KernelAbstractions GPU implementation

Completed answer for the exercice in `gray-scott.ipynb` §3.2–3.3 (`@kernel`, `@index`, `@private` registers, `@synchronize`, device `allocate` / launch / copy-back).



In [ ]:
# 5-point stencil Laplacian at grid point (x, y).
function laplacian(A, x, y, dx)
    @inbounds DAx = (A[x-1, y] - (2 * A[x, y]) + A[x+1, y]) / dx
    @inbounds DAy = (A[x, y-1] - (2 * A[x, y]) + A[x, y+1]) / dx
    return DAx + DAy
end



In [ ]:
# (1) @kernel — compiled for Backend
@kernel function calculate_gs_ka!(U, V, D_u, D_v, F, k, dt, dx, nx, ny, noise)
    # (3) register memory in advance via @private (per work-item)
    ΔU = @private Float64 1
    ΔV = @private Float64 1
    UVV = @private Float64 1
    Nr = @private Float64 1
    DU = @private Float64 1
    DV = @private Float64 1

    # (2) @index + ghost-cell guard (replaces the CPU for-loop)
    x, y = @index(Global, NTuple)[1:2]

    if (1 < x < nx) && (1 < y < ny)
        # U/V: device global memory; ΔU[1], …: registers
        ΔU[1] = laplacian(U, x, y, dx)
        ΔV[1] = laplacian(V, x, y, dx)

        UVV[1] = @inbounds U[x, y] * (V[x, y] ^ 2)
        Nr[1] = noise * randn(Float64)
        DU[1] = @inbounds D_u * ΔU[1] - UVV[1] + F * (1 - U[x, y]) + Nr[1]
        DV[1] = @inbounds D_v * ΔV[1] + UVV[1] - (F + k) * V[x, y]

        # (4) barriers around global writes
        @synchronize()
        @inbounds U[x, y] += DU[1] * dt
        @inbounds V[x, y] += DV[1] * dt
        @synchronize()
    end
end

function simulate_gs_ka(n = 500 #=Grid size=#, nsteps = 10000 #=Number of iterations=# ;
                        init_multi = 0.5,
                        noise = 0.0,
                        D_u = 0.16,
                        D_v = 0.08,
                        F = 0.035,
                        k = 0.06,
                        dt = 1.0,
                        dx = 1.0,
                        plot_every = 500)

    # Same host init as the reference CPU implementation
    U = rand(0.0:0.001:0.1, n, n)
    V = zeros(n, n)

    n_mid = div(n, 2)
    U[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= 1.0 * init_multi
    V[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= 0.5 * init_multi

    # (5) device global memory + host → device copy
    U_gpu = KernelAbstractions.allocate(Backend, Float64, n, n)
    V_gpu = KernelAbstractions.allocate(Backend, Float64, n, n)
    KernelAbstractions.copyto!(Backend, U_gpu, U)
    KernelAbstractions.copyto!(Backend, V_gpu, V)

    # (5) specialize @kernel for Backend
    kernel = calculate_gs_ka!(Backend, (cld(n, 16), cld(n, 16)))

    prog = Progress(div(nsteps, plot_every))
    plt = Plots.plot(title="Gray-Scott Model", xlabel="X", ylabel="Y", size=(600, 600))

    anim = Plots.@animate for t in 1:div(nsteps, plot_every)
        for _ in 1:plot_every
            # (5) launch with ndrange=(n, n)
            kernel(U_gpu, V_gpu, D_u, D_v, F, k, dt, dx, n, n, noise, ndrange=(n, n))
        end

        # (5) device → host + synchronize before plotting
        KernelAbstractions.copyto!(Backend, U, U_gpu)
        KernelAbstractions.synchronize(Backend)

        next!(prog)
        Plots.heatmap!(plt, U, color=cgrad([:black, :blue, :white]), clims=extrema(U), title="Gray-Scott Model")
    end
    gif(anim, "gs_ka.gif"; fps=10)
end



In [ ]:
@time simulate_gs_ka(500, 10000)



# 3. Experiments

Same parameter sweeps as in the main notebook.

In [ ]:
# Add some noise
@time simulate_gs_ka(500, 10000; noise=0.01)

In [ ]:
# Add a lot more noise
@time simulate_gs_ka(500, 10000; noise=0.03)

In [ ]:
# Tweak some parameters
@time simulate_gs_ka(500, 50000; k=.06054, F=.07528, noise=0.0001, init_multi=1.0, plot_every=2000)